# 05 - Cleaning Eurostat Long-Term Residents for Cameroonian Citizens

This notebook cleans Eurostat data on long-term resident permits granted to Cameroonian citizens.

Analytical role:

- supports Q3 on post-arrival trajectories
- captures one form of longer-term settlement after arrival
- standardizes the dataset for integration into `post_arrival_master.csv`

The indicator is interpreted separately from first permits, status changes and citizenship acquisition because each measure describes a different administrative event.


In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..")
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

EUROPE_PATH = DATA_RAW / "europe"
CLEANED_PATH = DATA_PROCESSED / "cleaned"

CLEANED_PATH.mkdir(parents=True, exist_ok=True)

In [ ]:
xls = pd.ExcelFile(EUROPE_PATH / "eurostat_long_term_residents_cameroon.xlsx")
xls.sheet_names

In [ ]:
sheet1_raw = pd.read_excel(
    EUROPE_PATH / "eurostat_long_term_residents_cameroon.xlsx",
    sheet_name="Sheet 1",
    header=None
)

sheet1_raw

In [ ]:
sheet1_raw = sheet1_raw.drop(
    columns=list(range(2, 21, 2)),
    errors="ignore"
)

sheet1_raw = sheet1_raw.drop(
    index=list(range(0, 9)) + list(range(46, 52)),
    errors="ignore"
)

sheet1_raw = sheet1_raw.reset_index(drop=True)

sheet1_raw

In [ ]:
sheet1_raw = sheet1_raw.drop(index=[1, 2, 3, 37], errors="ignore").reset_index(drop=True)
sheet1_raw

In [ ]:
sheet1_raw = sheet1_raw.replace(":", 0)
sheet1_raw

In [ ]:
# Use the first row as the header
sheet1_raw.columns = sheet1_raw.iloc[0]
sheet1_raw = sheet1_raw.iloc[1:].reset_index(drop=True)

# Rename TIME to destination_country
sheet1_raw = sheet1_raw.rename(
    columns={"TIME": "destination_country"}
)

# Reshape from wide format to long format
sheet1_raw = sheet1_raw.melt(
    id_vars=["destination_country"],
    var_name="year",
    value_name="residents"
)

# Clean data types
sheet1_raw["year"] = sheet1_raw["year"].astype(int)

sheet1_raw["residents"] = (
    pd.to_numeric(sheet1_raw["residents"], errors="coerce")
    .fillna(0)
    .astype(int)
)

# Reorder columns
sheet1_raw = sheet1_raw[
    ["destination_country", "year", "residents"]
]

sheet1_raw

In [ ]:
sheet1_raw["source"] = "eurostat"
sheet1_raw["nature"] = "long_term_resident"

def classify_covid_period(year):
    if year < 2020:
        return "pre_covid"
    elif year in [2020, 2021]:
        return "covid"
    else:
        return "post_covid"

sheet1_raw["covid_period"] = sheet1_raw["year"].apply(classify_covid_period)

sheet1_raw


In [ ]:
sheet1_raw = sheet1_raw.rename(
    columns={"permit": "permits"}
)

sheet1_raw


In [ ]:
sheet1_raw.to_csv(
    CLEANED_PATH / "eurostat_long_term_residents_cleaned.csv",
    index=False
)